# Anomaly Detection and Finetuning

The goal here is to train a Isolation Forest anomaly detection model and finetune a 3B-parameter LLM in order to detect satellite system abnormalities and deliver a report for mission ops teams.
The models will be trained using the [OPSSAT-AD](https://zenodo.org/records/12588359) dataset.

In [ ]:
# pip installs

### Carve out the train, validation, and test pool from the OPSSAT-AD dataset:

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("https://raw.githubusercontent.com/Tegh25/Satellite-Anomaly-Detector/refs/heads/main/model/dataset.csv")

test_df = df[df["train"] == 0]
train_pool = df[df["train"] == 1]

# Carve validation out of the train pool only, stratify so the ~20% anomaly rate
# is preserved in both slices, not skewed by chance
train_df, val_df = train_test_split(
    train_pool, test_size=0.15, stratify=train_pool["anomaly"], random_state=42
)

print(len(df), len(train_df), len(val_df), len(test_df))

2123 1354 240 529


In [2]:
print(df['gaps_squared'].equals(df['duration']))
match_count = (df['gaps_squared'] == df['duration']).sum()
print(match_count)

False
234
mean: median=4.278e-06, 90th pct=0.3401, max=1.119
var: median=6.26e-10, 90th pct=0.1267, max=0.2725
std: median=2.502e-05, 90th pct=0.356, max=0.522
kurtosis: median=-1.303, 90th pct=-0.5877, max=31.2
skew: median=0.0108, 90th pct=0.9237, max=4.375
n_peaks: median=1, 90th pct=2, max=37
smooth10_n_peaks: median=1, 90th pct=1, max=5
smooth20_n_peaks: median=1, 90th pct=2, max=4
diff_peaks: median=4, 90th pct=54, max=240
diff2_peaks: median=14, 90th pct=65, max=256
diff_var: median=3.62e-11, 90th pct=0.002364, max=0.08506
diff2_var: median=1.063e-11, 90th pct=0.0005733, max=0.2457
gaps_squared: median=551, 90th pct=2000, max=2.075e+04
var_div_duration: median=3.866e-12, 90th pct=0.0003967, max=0.001687
var_div_len: median=1.604e-11, 90th pct=0.001607, max=0.008031


### Anomaly Detection Model

Two models will be compared:
1. Isolation Forest (unsupervised)
2. Random Forest (supervised)

**Worth comparing against a supervised baseline.** Isolation Forest is unsupervised and it never uses the `anomaly` labels during training, only for threshold tuning afterward. Given this dataset has anomaly labels, a supervised model will likely outperform it directly. The unsupervised approach is still justified, like it can catch novel anomaly types a supervised model was never trained on.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_curve, classification_report

feature_cols = [
    "mean", "var", "std", "kurtosis", "skew",
    "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks",
    "diff_peaks", "diff2_peaks", "diff_var", "diff2_var",
    "gaps_squared", "var_div_duration", "var_div_len",
]

detector = IsolationForest(contamination=0.2, random_state=42)
detector.fit(train_df[feature_cols])

# Tune the decision threshold on validation instead of using the default
# the default threshold is derived purely from `contamination` at fit time,
# not from actually optimizing precision/recall on held-out data
val_scores = -detector.decision_function(val_df[feature_cols])
precision, recall, thresholds = precision_recall_curve(val_df["anomaly"], val_scores)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best_threshold = thresholds[f1[:-1].argmax()]

# Testing
test_scores = -detector.decision_function(test_df[feature_cols])
test_preds = (test_scores >= best_threshold).astype(int)
print("Isolation Forest (unsupervised):")
print(classification_report(test_df["anomaly"], test_preds))

Isolation Forest (unsupervised):
              precision    recall  f1-score   support

           0       0.87      0.49      0.62       416
           1       0.28      0.74      0.41       113

    accuracy                           0.54       529
   macro avg       0.58      0.61      0.52       529
weighted avg       0.75      0.54      0.58       529



In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
clf.fit(train_df[feature_cols], train_df["anomaly"])

sup_preds = clf.predict(test_df[feature_cols])
print("Random Forest (supervised):")
print(classification_report(test_df["anomaly"], sup_preds))

Random Forest (supervised):
              precision    recall  f1-score   support

           0       0.96      0.99      0.97       416
           1       0.95      0.83      0.89       113

    accuracy                           0.95       529
   macro avg       0.95      0.91      0.93       529
weighted avg       0.95      0.95      0.95       529



### Building the Finetuning Dataset

A frontier model will be used to generate high-quality draft reports for each labeled anomaly, conditioned on the real telemetry values and label. Later the smaller model will be trained on the high quality reports. This method is called distillation.

The below code block queries the backboard.io API and lists available OpenAI and Anthropic models.

In [9]:
import requests
from google.colab import userdata

BACKBOARD_API_KEY = userdata.get("BACKBOARD_API_KEY")
headers = {"X-API-Key": BACKBOARD_API_KEY}

models = requests.get(
    "https://app.backboard.io/api/models/provider/openai",
    headers=headers,
    params={"model_type": "llm", "limit": 25}
).json()

for m in models["models"]:
    print(f"{m['provider']}/{m['name']} — "
          f"ctx: {m['context_limit']}, "
          f"thinking: {m.get('supports_thinking', False)}, "
          f"in: ${m.get('input_cost_per_1m_tokens', 'N/A')}/M, "
          f"out: ${m.get('output_cost_per_1m_tokens', 'N/A')}/M")

models = requests.get(
    "https://app.backboard.io/api/models/provider/anthropic",
    headers=headers,
    params={"model_type": "llm", "limit": 25}
).json()

for m in models["models"]:
    print(f"{m['provider']}/{m['name']} — "
          f"ctx: {m['context_limit']}, "
          f"thinking: {m.get('supports_thinking', False)}, "
          f"in: ${m.get('input_cost_per_1m_tokens', 'N/A')}/M, "
          f"out: ${m.get('output_cost_per_1m_tokens', 'N/A')}/M")


openai/gpt-4.1 — ctx: 1047576, thinking: False, in: $2.0/M, out: $8.0/M
openai/gpt-4.1-mini — ctx: 1047576, thinking: False, in: $0.4/M, out: $1.6/M
openai/gpt-4.1-nano — ctx: 1047576, thinking: False, in: $0.1/M, out: $0.4/M
openai/gpt-4o — ctx: 128000, thinking: False, in: $2.5/M, out: $10.0/M
openai/gpt-4o-mini — ctx: 128000, thinking: False, in: $0.15/M, out: $0.6/M
openai/gpt-4o-mini-transcribe — ctx: 0, thinking: False, in: $1.25/M, out: $5.0/M
openai/gpt-4o-transcribe — ctx: 0, thinking: False, in: $2.5/M, out: $10.0/M
openai/gpt-4o-transcribe-diarize — ctx: 0, thinking: False, in: $2.5/M, out: $10.0/M
openai/gpt-5 — ctx: 400000, thinking: True, in: $1.25/M, out: $10.0/M
openai/gpt-5-chat-latest — ctx: 128000, thinking: False, in: $1.25/M, out: $10.0/M
openai/gpt-5-codex — ctx: 400000, thinking: True, in: $1.25/M, out: $10.0/M
openai/gpt-5-mini — ctx: 400000, thinking: True, in: $0.25/M, out: $2.0/M
openai/gpt-5-nano — ctx: 400000, thinking: True, in: $0.05/M, out: $0.4/M
openai

Reports will saved in a Google Drive file incrementally so that they are not lost in the case of an interruption.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

TEST_OUTPUT_PATH = '/content/drive/MyDrive/satellite-anomaly-project/test_reports.jsonl'

import os
os.makedirs(os.path.dirname(TEST_OUTPUT_PATH), exist_ok=True)

Mounted at /content/drive


#### Testing the Frontier Model's Reports

Generating effective reports for anomalous/non-anomalous rows is dependant on both data analysis and good prompt engineering. Below is a test that generate 10 reports on 5 anomalous and 5 nominal rows, the prompt and its included data has went through multiple iterations to ensure that the LLM output is effective. Sonnet 5 was chosen due to its quality and cost.

The report generation for all rows of the dataset were done locally using a python script, not in this Colab file.

In [6]:
import json
import requests
from google.colab import userdata

# Keys and config
BACKBOARD_API_KEY = userdata.get("BACKBOARD_API_KEY")
BACKBOARD_API_URL = "https://app.backboard.io/api"
BACKBOARD_LLM_PROVIDER = "anthropic"
BACKBOARD_MODEL_NAME = "claude-sonnet-5"

def load_completed_ids(path):
    completed = set()
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f:
                completed.add(json.loads(line)['segment'])
    return completed

def append_result(path, segment_id, report):
    with open(path, 'a') as f:
        f.write(json.dumps({"segment": segment_id, "report": report}) + "\n")

completed_ids = load_completed_ids(TEST_OUTPUT_PATH)
print(f"Already done: {len(completed_ids)} rows")

## Test of append result function ##
# report = "Segment statistics are consistent with nominal channel behavior: near-zero mean with very low variance/std dev indicates a stable, low-amplitude signal with no significant excursions. The negative kurtosis (-1.275) suggests a flatter-than-Gaussian distribution rather than spiky outliers, and low skew confirms symmetry — no evidence of transient spikes or sudden jumps. Peak counts remain consistent (1) across raw and smoothed views, and while sampling gap irregularity (1150) is noted, it does not manifest in derivative statistics as anomalous discontinuities, suggesting it's a minor timing artifact rather than a data-loss event. **No action required** — continue routine monitoring."
# append_result(TEST_OUTPUT_PATH, 789, report)

if not BACKBOARD_API_KEY:
    raise ValueError("Set the BACKBOARD_API_KEY environment variable before running this notebook cell.")

feature_cols = [
    "mean", "var", "std", "kurtosis", "skew",
    "n_peaks", "smooth10_n_peaks", "smooth20_n_peaks",
    "diff_peaks", "diff2_peaks", "diff_var", "diff2_var",
    "gaps_squared", "var_div_duration", "var_div_len",
]

# Compute reference statistics from train_df
reference_stats = train_df[feature_cols].describe(percentiles=[0.5, 0.9]).loc[["50%", "90%", "max"]]

def format_reference_line(col):
    return f"{col}: median={reference_stats.loc['50%', col]:.4g}, 90th pct={reference_stats.loc['90%', col]:.4g}, max={reference_stats.loc['max', col]:.4g}"

reference_block = "\n".join(format_reference_line(c) for c in feature_cols)

def generate_report(row):
    prompt = f"""You are a spacecraft operations engineer reviewing a flagged telemetry segment.
Note: these are pre-computed statistical features of the segment, not raw sensor values —
reason about what they imply rather than treating them as physical readings.

For internal calibration only — DO NOT mention these numbers, percentiles, or "the dataset" in your
final report. Use them only to judge whether a statistic for this segment is genuinely unusual:
{reference_block}

Write the report as a standalone assessment of this segment, in the same plain language a mission-ops
engineer would use looking at one live reading with no visibility into a historical dataset. Do not
reference medians, percentiles, "typical values," or any dataset-level comparison directly — describe
what's unusual in physical/operational terms instead (e.g. "unusually high for this channel," not
"90th-percentile value").

Channel: {row['channel']}
Segment length: {row['len']} samples over {row['duration']} time units
Mean: {row['mean']:.6g}, Std dev: {row['std']:.6g}, Variance: {row['var']:.6g}
Kurtosis: {row['kurtosis']:.3f}, Skew: {row['skew']:.3f}  (distribution shape — spiky/flat, lopsided/symmetric)
Peak count (raw / smoothed-10 / smoothed-20): {row['n_peaks']} / {row['smooth10_n_peaks']} / {row['smooth20_n_peaks']}
Derivative peak count (1st / 2nd diff): {row['diff_peaks']} / {row['diff2_peaks']}
Derivative variance (1st / 2nd diff): {row['diff_var']:.6g} / {row['diff2_var']:.6g}
Sampling gap irregularity: {row['gaps_squared']}
Label: {"ANOMALY" if row['anomaly'] == 1 else "NOMINAL"}

First, identify which of the provided statistics are actually the most unusual or informative
for THIS segment specifically, do not default to any one feature (e.g. sampling gap irregularity)
unless it is genuinely the standout signal here. Different segments may be explained by different
features: distribution shape, peak structure, derivative behavior, or sampling irregularity can each
be the dominant story depending on the row.

Write a concise mission-ops report (3-4 sentences) covering:
- Which specific statistic(s) actually drive your assessment for this segment, and why
- The likely cause, reasoned from those specific statistics (not a generic category)
- Suggested next step for the ops team, specific to that likely cause

(Remember: describe what's unusual in plain operational terms, with no mention of percentiles,
medians, or dataset-level comparisons.)

If NOMINAL, write a brief confirmation grounded in the specific statistics, not a generic template.

Respond with plain prose only, no headers or markdown formatting. No em dashes."""

    response = requests.post(
        f"{BACKBOARD_API_URL}/threads/messages",
        headers={"X-API-Key": BACKBOARD_API_KEY},
        json={
            "content": prompt,
            "llm_provider": BACKBOARD_LLM_PROVIDER,
            "model_name": BACKBOARD_MODEL_NAME,
            "stream": False,
            "memory": "off",
            "system_prompt": "You are a spacecraft operations engineer. Provide concise, evidence-based assessments for telemetry anomalies.",
        },
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    return payload.get("content") or payload.get("message") or ""

anomaly_sample = df[df['anomaly'] == 1].sample(n=5, random_state=42)
nominal_sample = df[df['anomaly'] == 0].sample(n=5, random_state=42)
stratified_sample = pd.concat([anomaly_sample, nominal_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(stratified_sample[['segment', 'channel', 'anomaly']])

for _, row in stratified_sample.iterrows():
    if row['segment'] in completed_ids:
        continue  # skip rows already generated on a prior run
    try:
        report = generate_report(row)
        append_result(TEST_OUTPUT_PATH, row['segment'], report)
    except Exception as e:
        print(f"Row {row['segment']} failed: {e}")
        continue  # move on instead of crashing the whole loop

Already done: 0 rows
   segment   channel  anomaly
0     1808  CADC0872        0
1      235  CADC0892        1
2     1280  CADC0873        0
3     1183  CADC0873        1
4     1022  CADC0873        0
5      293  CADC0874        1
6     1035  CADC0873        0
7     1617  CADC0888        1
8      945  CADC0892        1
9     2055  CADC0872        0
